Ce projet a pour objectif de prédire l’issue d’un match de tennis professionnel en utilisant des données historiques ATP/WTA et des techniques avancées de machine learning.
L’enjeu est de modéliser la probabilité de victoire d’un joueur en s’appuyant sur des indicateurs de performance construits à partir de son historique.

Le travail s’articule autour de plusieurs étapes :

collecte et préparation des données, incluant le nettoyage, la structuration chronologique et la gestion des valeurs manquantes

construction de features pour capturer la dynamique réelle des joueurs

entraînement de plusieurs modèles (régression logistique, Random Forest, XGBoost)

évaluation des performances (AUC, accuracy, F1-score, log loss, Brier Score)

interprétation du modèle final pour comprendre les facteurs déterminants d’une victoire

L’objectif final est de proposer un modèle robuste, interprétable et performant, capable d’estimer la probabilité de victoire d’un joueur avant un match.


In [248]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import kagglehub
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    accuracy_score, f1_score, confusion_matrix,
    precision_recall_curve, log_loss, brier_score_loss
)
plt.style.use('seaborn-v0_8-whitegrid')
path = kagglehub.dataset_download("dissfya/atp-tennis-2000-2023daily-pull")
file_path = os.path.join(path, "atp_tennis.csv")
df = pd.read_csv(file_path)


Using Colab cache for faster access to the 'atp-tennis-2000-2023daily-pull' dataset.


Cette fonction permet de transformer une chaîne de score brute (ex : "6-4 3-6 7-6") en indicateurs numériques exploitables pour l’analyse.
Elle nettoie le format du score, découpe les sets, et identifie les tie-breaks.

In [249]:
def parse_score(score):
    if pd.isna(score):
        return {
            "Sets_Played": 0,
            "TB_Played": 0
        }

    cleaned = score.replace("–", "-").replace("—", "-").replace("‑", "-")

    sets = cleaned.split()

    sets_played = len(sets)
    tb_played = 0

    for s in sets:
        if "-" not in s:
            continue

        g1, g2 = map(int, s.split("-"))

        # Tie-break détecté : 7-6 ou 6-7
        if (g1 == 7 and g2 == 6) or (g1 == 6 and g2 == 7):
            tb_played += 1

    return {
        "Sets_Played": sets_played,
        "TB_Played": tb_played
    }
parsed = df["Score"].apply(parse_score).apply(pd.Series)
df = pd.concat([df, parsed], axis=1)


In [250]:
df

,Tournament,Date,Series,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score,Sets_Played,TB_Played
0,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Dosedel S.,Ljubicic I.,Dosedel S.,63,77,-1,-1,-1.00,-1.00,6-4 6-2,2,0
1,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Clement A.,Enqvist T.,Enqvist T.,56,5,-1,-1,-1.00,-1.00,3-6 3-6,2,0
2,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Escude N.,Baccanello P.,Escude N.,40,655,-1,-1,-1.00,-1.00,6-7 7-5 6-3,3,1
3,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Knippschild J.,Federer R.,Federer R.,87,65,-1,-1,-1.00,-1.00,1-6 4-6,2,0
4,Australian Hardcourt Championships,2000-01-03,International,Outdoor,Hard,1st Round,3,Fromberg R.,Woodbridge T.,Fromberg R.,81,198,-1,-1,-1.00,-1.00,7-6 5-7 6-4,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66676,Masters Cup,2025-11-14,Masters Cup,Indoor,Hard,Round Robin,3,Sinner J.,Shelton B.,Sinner J.,2,5,10000,3970,1.05,10.00,6-3 7-6,2,1
66677,Masters Cup,2025-11-14,Masters Cup,Indoor,Hard,Round Robin,3,Zverev A.,Auger-Aliassime F.,Auger-Aliassime F.,3,8,4960,3845,4.50,1.20,4-6 6-7,2,1
66678,Masters Cup,2025-11-15,Masters Cup,Indoor,Hard,Semifinals,3,Sinner J.,De Minaur A.,Sinner J.,2,7,10000,3935,1.05,10.00,7-5 6-2,2,0
66679,Masters Cup,2025-11-15,Masters Cup,Indoor,Hard,Semifinals,3,Auger-Aliassime F.,Alcaraz C.,Alcaraz C.,8,1,3845,11050,4.50,1.20,2-6 4-6,2,0


Transformation du dataset brut (où chaque ligne représente un match avec deux joueurs) en un format long, où chaque ligne représente un joueur dans un match.
Cela permet de calculer des statistiques historiques cohérentes pour chaque joueur avant chaque rencontre.

In [251]:
df_long = pd.DataFrame({
    "Player": pd.concat([df["Player_1"], df["Player_2"]]),
    "Opponent": pd.concat([df["Player_2"], df["Player_1"]]),
    "Date": pd.concat([df["Date"], df["Date"]]),
    "odd_1": pd.concat([df["Odd_1"], df["Odd_1"]]),
    "odd_2": pd.concat([df["Odd_2"], df["Odd_2"]]),
    "Surface": pd.concat([df["Surface"], df["Surface"]]),
    "Best_of": pd.concat([df["Best of"], df["Best of"]]),
    "Rank": pd.concat([df["Rank_1"], df["Rank_2"]]),
    "OppRank": pd.concat([df["Rank_2"], df["Rank_1"]]),
    "Won": pd.concat([
        (df["Winner"] == df["Player_1"]).astype(int),
        (df["Winner"] == df["Player_2"]).astype(int)
    ]),
})
df_long = df_long.reset_index(drop=True)
df_p = df_long.copy()
df_p = df_p.sort_values(["Player", "Date"])
df_p["Matches_Before"] = df_p.groupby("Player").cumcount()
df_p["Wins_Before"] = df_p.groupby("Player")["Won"].shift().fillna(0).groupby(df_p["Player"]).cumsum()
df_p["WR_All"] = df_p["Wins_Before"] / df_p["Matches_Before"]
df_p["WR_All"] = df_p["WR_All"].replace([np.inf, -np.inf], np.nan)
df_p["WR_Last5"] = df_p.groupby("Player")["Won"].shift().rolling(window=5, min_periods=1).mean()
df_p["WR_Last100"] = df_p.groupby("Player")["Won"].shift().rolling(window=100, min_periods=1).mean()
df_p = df_p.sort_index()


df_o = df_long.copy()
df_o = df_o.sort_values(["Opponent", "Date"])
df_o["OppWon"] = 1 - df_o["Won"]
df_o["Opp_Matches_Before"] = df_o.groupby("Opponent").cumcount()
df_o["Opp_Wins_Before"] = df_o.groupby("Opponent")["OppWon"].shift().fillna(0).groupby(df_o["Opponent"]).cumsum()
df_o["Opp_WR_All"] = df_o["Opp_Wins_Before"] / df_o["Opp_Matches_Before"]
df_o["Opp_WR_All"] = df_o["Opp_WR_All"].replace([np.inf, -np.inf], np.nan)
df_o["Opp_WR_Last5"] = df_o.groupby("Opponent")["OppWon"].shift().rolling(window=5, min_periods=1).mean()
df_o["Opp_WR_Last100"] = df_o.groupby("Opponent")["OppWon"].shift().rolling(window=100, min_periods=1).mean()
df_o = df_o.sort_index()
df_p = df_p.loc[df_long.index]
df_o = df_o.loc[df_long.index]


player_features = ["WR_All","WR_Last5","WR_Last100",  "Matches_Before"]
opp_features = ["Opp_WR_All", "Opp_WR_Last5","Opp_WR_Last100","Opp_Matches_Before"]
df_long[player_features] = df_p[player_features].values
df_long[opp_features] = df_o[opp_features].values
df_long["Rank_Ratio"] = df_long["Rank"] / (df_long["OppRank"])
df_long["WR_Diff"] = df_long["WR_All"] - df_long["Opp_WR_All"]
df_long["WR_Ratio"] = df_long["WR_All"] / (df_long["Opp_WR_All"] )
df_long["Momentum_Diff"] = df_long["WR_Last5"] - df_long["Opp_WR_Last5"]
df_long["Momentum_Last100_Diff"] = df_long["WR_Last100"] - df_long["Opp_WR_Last100"]
df_long["Experience_Diff"] = df_long["Matches_Before"] - df_long["Opp_Matches_Before"]
df_long["Date"] = pd.to_datetime(df_long["Date"])
df_long["Year"] = df_long["Date"].dt.year
df_long["Month"] = df_long["Date"].dt.month


df_surf = df_long.copy()
df_surf = df_surf.sort_values(["Player", "Surface", "Date"])
df_surf["Wins_Surface_Before"] = df_surf.groupby(["Player", "Surface"])["Won"].shift().fillna(0).groupby([df_surf["Player"], df_surf["Surface"]]).cumsum()
df_surf["Matches_Surface_Before"] = df_surf.groupby(["Player", "Surface"]).cumcount()
df_surf["WR_Surface"] = df_surf["Wins_Surface_Before"] / df_surf["Matches_Surface_Before"]
df_surf["WR_Surface"] = df_surf["WR_Surface"].replace([np.inf, -np.inf], np.nan)
df_surf = df_surf.sort_index()
df_long["WR_Surface"] = df_surf["WR_Surface"].values


Préparation du dataset pour l’entraînement des modèles de machine learning.
filtrage des joueurs, encodage des variables catégorielles, sélection des features pertinentes et traitement des valeurs manquantes.

In [252]:
df_model = df_long.copy()
df_model = df_model.groupby("Player").filter(lambda x: len(x) >= 5)

le_surface = LabelEncoder()
df_model["Surface_Encoded"] = le_surface.fit_transform(df_model["Surface"])

feature_cols =  [
    "WR_All", "Opp_WR_All", "WR_Diff",

    "WR_Last5", "Opp_WR_Last5", "Momentum_Diff",

    "WR_Last100","Opp_WR_Last100","Momentum_Last100_Diff",

    "WR_Surface", "Surface_Encoded", "Experience_Diff",

    "Rank", "OppRank",  "Rank_Ratio",

    "Best_of","Month","Year",

    "Matches_Before","Opp_Matches_Before"


]
for col in feature_cols:
    df_model[col] = df_model[col].replace([np.inf, -np.inf], np.nan)
    df_model[col] = df_model[col].fillna(df_model[col].median())

X = df_model[feature_cols]
y = df_model["Won"]

df_model = df_model.sort_values('Date')
split = int(len(df_model) * 0.8)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)




 Régression logistique :

In [253]:
from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV


lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)
f1_lr = f1_score(y_test, y_pred_lr)


print(f"LR — Acc:{acc_lr:.3f} | AUC:{auc_lr:.3f} | F1:{f1_lr:.3f}")
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]
lrlogloss=log_loss(y_test, y_proba_lr)
print("log_loss:",lrlogloss)

brier_lr = brier_score_loss(y_test, y_proba_lr)

print("Brier score :", brier_lr)


LR — Acc:0.652 | AUC:0.712 | F1:0.657
log_loss: 0.6198726386528118
Brier score : 0.2156088408981323


Random Forest :

In [254]:

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)
f1_rf = f1_score(y_test, y_pred_rf)
rflogloss=log_loss(y_test, y_proba_rf)
brier_rf = brier_score_loss(y_test, y_proba_rf)

print("Brier score :", brier_rf)
print("log_loss:",rflogloss)

print(f"RF — Acc:{acc_rf:.3f} | AUC:{auc_rf:.3f} | F1:{f1_rf:.3f}")


Brier score : 0.21173586440705944
log_loss: 0.6100823483489533
RF — Acc:0.656 | AUC:0.723 | F1:0.656


XGBoost :

In [255]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

acc_xgb = accuracy_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_proba_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
proba = xgb_model.predict_proba(X_test)[:, 1]
ll = log_loss(y_test, proba)
print("Log Loss pour XGB:", ll)
print(f"XGB — Acc:{acc_xgb:.3f} | AUC:{auc_xgb:.3f} | F1:{f1_xgb:.3f}")
brier_xgb = brier_score_loss(y_test, y_proba_xgb)

print("Brier score :", brier_xgb)


Log Loss pour XGB: 0.6078362511505857
XGB — Acc:0.657 | AUC:0.724 | F1:0.660
Brier score : 0.2109299683763912


In [256]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [acc_lr, acc_rf, acc_xgb],
    "AUC": [auc_lr, auc_rf, auc_xgb],
    "F1": [f1_lr, f1_rf, f1_xgb],
    "log_loss":[lrlogloss,rflogloss,ll],
    "Brier Score": [brier_lr, brier_rf, brier_xgb]

})

print(results.sort_values("AUC", ascending=False))


                 Model  Accuracy       AUC        F1  log_loss  Brier Score
2              XGBoost  0.657393  0.724492  0.659945  0.607836     0.210930
1        Random Forest  0.655953  0.722527  0.656265  0.610082     0.211736
0  Logistic Regression  0.651594  0.712033  0.656631  0.619873     0.215609


Les trois modèles testés — XGBoost, Random Forest et Régression Logistique — présentent des performances globalement proches, mais avec des différences importantes selon les métriques.
Le XGBoost obtient la meilleure AUC (0.724), ce qui signifie qu’il discrimine légèrement mieux les matchs gagnés des matchs perdus.

Même si l’écart est modeste, c’est un signal clair que XGBoost capture mieux les interactions non linéaires entre variables.

Le XGBoost obtient le meilleur log loss (0.607), ce qui signifie que ses probabilités sont les plus informatives parmi les trois modèles.

In [257]:

from sklearn.inspection import permutation_importance



importance_gain = xgb_model.get_booster().get_score(importance_type='gain')
importance_gain = pd.DataFrame.from_dict(importance_gain, orient='index', columns=['gain'])
importance_gain = importance_gain.sort_values('gain', ascending=False)

print("Importance interne (gain) :")
print(importance_gain)

perm = permutation_importance(
    xgb_model, X_test, y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_brier_score'
)

perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

print("\nPermutation importance :")
print(perm_df)


Importance interne (gain) :
                            gain
Rank_Ratio             85.897339
Momentum_Last100_Diff  29.264053
WR_Diff                27.151827
Momentum_Diff          10.655050
OppRank                10.555484
Best_of                10.009403
WR_Surface              8.398490
Rank                    8.288481
Matches_Before          7.541437
Opp_Matches_Before      7.347686
Opp_WR_Last5            6.830331
Opp_WR_All              6.771111
Year                    6.379757
Opp_WR_Last100          6.082603
Experience_Diff         6.071411
WR_All                  5.879714
Surface_Encoded         5.855887
WR_Last100              5.708937
WR_Last5                5.078436
Month                   4.760289

Permutation importance :
                  feature  importance_mean  importance_std
14             Rank_Ratio         0.011031        0.000486
9              WR_Surface         0.006514        0.000326
8   Momentum_Last100_Diff         0.002551        0.000244
2                

Rank_Ratio domine largement. C’est la variable la plus informative du modèle, et de très loin. Elle capture un rapport de force global qui semble résumer beaucoup de choses sur le niveau des joueurs. Ensuite, on retrouve des indicateurs liés à la forme récente (Momentum_Last100_Diff, WR_Diff) et à la surface (WR_Surface), qui ressortent clairement dans la permutation importance. Ces variables influencent directement la calibration du modèle, ce qui montre qu’il s’appuie bien sur des signaux pertinents et récents.

In [258]:
df_test = df_model.iloc[split:].copy()
df_test["y_true"] = y_test.values
df_test["proba"] = y_proba_xgb
df_test["year"] = df_test["Date"].dt.year
df_test.groupby("year").apply(
    lambda x: brier_score_loss(x["y_true"], x["proba"])
)


/tmp/ipython-input-2649511186.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test.groupby("year").apply(


,0
year,
2020,0.189941
2021,0.199113
2022,0.217369
2023,0.216242
2024,0.212499
2025,0.214229


Même si l’on observe de légères fluctuations d’une année à l’autre, le Brier Score reste dans une fourchette étroite, comprise entre 0.19 et 0.21, 2O2O est performant probablement à cause du covid qui a reduit le calendrier.

In [259]:



df_test.groupby("Surface").apply(
    lambda x: brier_score_loss(x["y_true"], x["proba"])
)


/tmp/ipython-input-983270375.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test.groupby("Surface").apply(


,0
Surface,
Clay,0.212606
Grass,0.210600
Hard,0.210098


Ces valeurs montrent que le Brier Score du modèle reste très stable selon la surface, avec des écarts minimes entre terre battue, gazon et dur.

In [260]:
df_test["decile"] = pd.qcut(df_test["proba"], 10)

df_test.groupby("decile").agg(
    mean_proba=("proba", "mean"),
    win_rate=("y_true", "mean"),
    count=("y_true", "size")
)


/tmp/ipython-input-1077746473.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_test.groupby("decile").agg(


,mean_proba,win_rate,count
decile,,,
"(0.009600000000000001, 0.231]",0.152990,0.155362,2639
"(0.231, 0.325]",0.280006,0.288855,2638
"(0.325, 0.395]",0.361774,0.372631,2638
"(0.395, 0.45]",0.423290,0.439727,2638
"(0.45, 0.502]",0.476215,0.472906,2639
"(0.502, 0.553]",0.527752,0.518196,2638
"(0.553, 0.612]",0.581652,0.572024,2638
"(0.612, 0.683]",0.645850,0.639121,2638
"(0.683, 0.776]",0.726536,0.723654,2638


Les déciles de probabilité montrent que le modèle produit des prédictions cohérentes : lorsque la probabilité moyenne augmente, le taux de victoire observé augmente lui aussi de manière régulière. Les écarts entre mean_proba et win_rate restent faibles dans l’ensemble, ce qui indique une calibration correcte.

In [261]:
proba_clip = np.clip(y_proba_xgb, 0.05, 0.95)
brier_clip = brier_score_loss(y_test, proba_clip)
print("Brier score après clipping :", brier_clip)

Brier score après clipping : 0.21092474316410778


Le Brier Score après clipping reste pratiquement identique, autour de 0.2123, ce qui montre que le modèle ne souffre pas de probabilités extrêmes problématiques. Le clipping n’apporte donc aucun gain mesurable, signe que la distribution initiale des probabilités était déjà stable et correctement maîtrisée

In [262]:
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score, f1_score

df_naive = df_model.loc[y_test.index].copy()

df_naive["Naive_Prob"] = (df_naive["Rank"] < df_naive["OppRank"]).astype(float)

df_naive["Naive_Pred"] = (df_naive["Naive_Prob"] > 0.5).astype(int)

acc_naive = accuracy_score(df_naive["Won"], df_naive["Naive_Pred"])
auc_naive = roc_auc_score(df_naive["Won"], df_naive["Naive_Prob"])
f1_naive = f1_score(df_naive["Won"], df_naive["Naive_Pred"])
brier_naive = brier_score_loss(df_naive["Won"], df_naive["Naive_Prob"])

print(f"Modèle naïf — Acc:{acc_naive:.3f} | AUC:{auc_naive:.3f} | F1:{f1_naive:.3f} | Brier:{brier_naive:.4f}")


Modèle naïf — Acc:0.645 | AUC:0.645 | F1:0.648 | Brier:0.3554


Le modèle naïf affiche des performances nettement inférieures à celles des modèles supervisés.

In [263]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, brier_score_loss

df_model["odd_1"] = pd.to_numeric(df_model["odd_1"], errors="coerce")
df_model["odd_2"] = pd.to_numeric(df_model["odd_2"], errors="coerce")

df_model["odd_1"] = df_model["odd_1"].mask(df_model["odd_1"] <= 0, np.nan)
df_model["odd_2"] = df_model["odd_2"].mask(df_model["odd_2"] <= 0, np.nan)

df_book = df_model.dropna(subset=["odd_1", "odd_2"]).copy()

p1_raw = 1 / df_book["odd_2"]
p2_raw = 1 / df_book["odd_1"]

overround = p1_raw + p2_raw
df_book["Book_Prob"] = p1_raw / overround

common_idx = df_book.index.intersection(y_test.index)

y_true_book = y_test.loc[common_idx]
y_prob_book = df_book.loc[common_idx, "Book_Prob"]

y_pred_book = (y_prob_book > 0.5).astype(int)

acc_book = accuracy_score(y_true_book, y_pred_book)
auc_book = roc_auc_score(y_true_book, y_prob_book)
f1_book = f1_score(y_true_book, y_pred_book)
brier_book = brier_score_loss(y_true_book, y_prob_book)

print(f"Bookmaker — Acc:{acc_book:.3f} | AUC:{auc_book:.3f} | F1:{f1_book:.3f} | Brier:{brier_book:.4f}")


Bookmaker — Acc:0.685 | AUC:0.758 | F1:0.683 | Brier:0.1998


Les résultats du bookmaker montrent qu’il fait clairement mieux que les modèles supervisés. Avec une accuracy de 0.685 et une AUC de 0.758, ses probabilités arrivent à bien classer les matchs et à repérer les favoris de manière efficace.

In [265]:


i = np.random.randint(0, len(X_test))
match_features = X_test.iloc[[i]]

match_info = df_test.iloc[i]
proba = xgb_model.predict_proba(match_features)[0, 1]
pred = int(proba >= 0.5)
true = y_test.iloc[i]
print("=== Match aléatoire ===")
print(f"Index : {i}")
print("\n--- Infos du match ---")
print(match_info)

print("\n--- Prédiction du modèle ---")
print(f"Probabilité de victoire : {proba:.3f}")
print(f"Prédiction : {pred}")
print(f"Résultat réel : {true}")

verdict = "✔️ Correct" if pred == true else "❌ Incorrect"
print(f"\nVerdict : {verdict}")


=== Match aléatoire ===
Index : 18466

--- Infos du match ---
Player                                     Monteiro T.
Opponent                                    Monfils G.
Date                               2024-05-08 00:00:00
odd_1                                              2.2
odd_2                                             1.67
Surface                                           Clay
Best_of                                              3
Rank                                               106
OppRank                                             38
Won                                                  1
WR_All                                        0.404624
WR_Last5                                           0.6
WR_Last100                                        0.43
Matches_Before                                   173.0
Opp_WR_All                                    0.642066
Opp_WR_Last5                                       0.4
Opp_WR_Last100                                    0.59
Opp